In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Load the merged dataset from notebook 01
fs = pd.read_csv('../data/OASIS3_Freesurfer_output.csv')
demo = pd.read_csv('../data/OASIS3_demographics.csv')
cdr = pd.read_csv('../data/OASIS3_UDSb4_cdr.csv')

# Rebuild merged dataset
cdr_max = cdr.groupby('OASISID')['CDRTOT'].max().reset_index()
cdr_max.columns = ['OASISID', 'max_CDR']
cdr_max['diagnosis_group'] = cdr_max['max_CDR'].apply(
    lambda x: 'CN' if x==0 else ('MCI' if x==0.5 else 'AD'))

fs_first = fs.sort_values('MR_session').groupby('Subject').first().reset_index()
fs_first = fs_first.rename(columns={'Subject': 'OASISID'})

df = demo.merge(cdr_max, on='OASISID')
df = df.merge(fs_first[['OASISID', 'TOTAL_HIPPOCAMPUS_VOLUME',
                          'lh_entorhinal_thickness', 'rh_entorhinal_thickness',
                          'lh_precuneus_thickness', 'rh_precuneus_thickness',
                          'IntraCranialVol', 'FS QC Status']], on='OASISID')
df = df[df['FS QC Status'] == 'Passed']
df['hippo_ICV_corrected'] = (df['TOTAL_HIPPOCAMPUS_VOLUME'] / df['IntraCranialVol']) * 100
df['mean_entorhinal_thickness'] = (df['lh_entorhinal_thickness'] + 
                                    df['rh_entorhinal_thickness']) / 2

# Statistical tests
features = {
    'Hippocampal Volume (ICV-corrected)': 'hippo_ICV_corrected',
    'Entorhinal Thickness': 'mean_entorhinal_thickness',
    'Precuneus Thickness (LH)': 'lh_precuneus_thickness'
}

order = ['CN', 'MCI', 'AD']

print("=" * 60)
print("KRUSKAL-WALLIS TEST (overall group differences)")
print("=" * 60)

for label, col in features.items():
    groups = [df[df['diagnosis_group'] == g][col].dropna() for g in order]
    stat, p = kruskal(*groups)
    print(f"\n{label}:")
    print(f"  H-statistic: {stat:.2f}, p-value: {p:.2e}")

print("\n" + "=" * 60)
print("MANN-WHITNEY U TESTS (pairwise, Bonferroni corrected)")
print("=" * 60)

n_comparisons = 3  # CN-MCI, CN-AD, MCI-AD
alpha = 0.05 / n_comparisons  # Bonferroni correction

for label, col in features.items():
    print(f"\n{label}:")
    for g1, g2 in combinations(order, 2):
        d1 = df[df['diagnosis_group'] == g1][col].dropna()
        d2 = df[df['diagnosis_group'] == g2][col].dropna()
        stat, p = mannwhitneyu(d1, d2, alternative='two-sided')
        sig = "***" if p < alpha else "ns"
        print(f"  {g1} vs {g2}: U={stat:.0f}, p={p:.2e} {sig}")

KRUSKAL-WALLIS TEST (overall group differences)

Hippocampal Volume (ICV-corrected):
  H-statistic: 247.72, p-value: 1.62e-54

Entorhinal Thickness:
  H-statistic: 117.22, p-value: 3.52e-26

Precuneus Thickness (LH):
  H-statistic: 115.89, p-value: 6.85e-26

MANN-WHITNEY U TESTS (pairwise, Bonferroni corrected)

Hippocampal Volume (ICV-corrected):
  CN vs MCI: U=86695, p=6.16e-20 ***
  CN vs AD: U=83239, p=7.28e-47 ***
  MCI vs AD: U=22863, p=1.24e-10 ***

Entorhinal Thickness:
  CN vs MCI: U=76978, p=1.03e-08 ***
  CN vs AD: U=73148, p=9.47e-25 ***
  MCI vs AD: U=20655, p=2.35e-05 ***

Precuneus Thickness (LH):
  CN vs MCI: U=76148, p=5.51e-08 ***
  CN vs AD: U=73100, p=1.16e-24 ***
  MCI vs AD: U=21159, p=2.22e-06 ***


## Statistical Interpretation

All three regions show highly significant differences across diagnostic groups 
(Kruskal-Wallis, all p < 10e-25). Pairwise Mann-Whitney U tests with Bonferroni 
correction confirm significant separation between every group pair for all regions.

Hippocampal volume shows the strongest effect (H=247.72, p=1.62e-54), consistent 
with its established role as the primary structural biomarker of AD progression. 
Critically, MCI vs AD separation is also significant for all regions, confirming 
these measures track disease severity rather than simply distinguishing normal 
from pathological.

The entorhinal cortex and precuneus show comparable effect sizes to each other 
(H≈116-117), suggesting both contribute independent discriminative information 
beyond hippocampal volume alone — motivating a multivariate classification 
approach in the next analysis.

In [2]:
# Cohen's d effect sizes for CN vs AD
def cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    pooled_std = np.sqrt(((n1-1)*g1.std()**2 + (n2-1)*g2.std()**2) / (n1+n2-2))
    return abs(g1.mean() - g2.mean()) / pooled_std

print("Cohen's d effect sizes (CN vs AD):")
for label, col in features.items():
    cn = df[df['diagnosis_group']=='CN'][col].dropna()
    ad = df[df['diagnosis_group']=='AD'][col].dropna()
    d = cohens_d(cn, ad)
    magnitude = 'large' if d > 0.8 else ('medium' if d > 0.5 else 'small')
    print(f"  {label}: d={d:.3f} ({magnitude})")
    

Cohen's d effect sizes (CN vs AD):
  Hippocampal Volume (ICV-corrected): d=1.583 (large)
  Entorhinal Thickness: d=1.161 (large)
  Precuneus Thickness (LH): d=1.029 (large)
